# Step 09.2 — Vanishing and Exploding Gradients：长期 Credit Assignment 为什么会失真

Step 9.1 已经得到长期依赖中的核心结构。这里先限定讨论对象：**如果 loss 只定义在末态，或者我们只追踪“未来 loss 沿 recurrent chain 回传到早期状态”的那一条 path contribution**，则可以写成：

$
g_t^{\text{future}}
=
J_{t+1,t}^T
J_{t+2,t+1}^T
\cdots
J_{T-1,T-2}^T
g_{T-1}.
$

如果总 loss 是 $L=\sum_{t=0}^{T-1}\ell_t$、每个时间步都有 local loss，那么某个 hidden state 的**总梯度**还必须加上它自己的 local-loss contribution；Section 3 会把这两项分开写清楚。

本节只回答：**这些沿 recurrent path 连续相乘的局部导数 / Jacobian，会怎样改变 future-path gradient contribution 的大小和方向？**

继续沿用 9.1 的统一记号：

```text
T = sequence length
t = 0,1,...,T-1
h_-1 = initial hidden state
h_(T-1) = final hidden state
H = RNN hidden-state width
```

本节五部分：① 用 `0.5^n / 1.5^n` 建立重复乘法直觉；② 接回 tanh RNN，解释 saturation 与 hidden state 为什么改变局部因子；③ 升级到 vector/Jacobian，说明方向为何决定长期 gradient，不能把 scalar 结论直接套到所有矩阵；④ 首次定义 gradient norm 并解释 gradient clipping；⑤ 总结 clipping 的能力边界与长期 credit assignment 的真正困难。

这里只准备 Step 9.2 材料，不进入 Step 9.3。


## 1. 从最小直觉开始：为什么 `0.5^n` 会消失，`1.5^n` 会爆炸？

先暂时不看 RNN，只研究最简单的 chain rule。

假设一个早期变量到最终 loss 之间要经过 n 个局部步骤，并且为了教学，把每一步的局部导数都假设成同一个 scalar `c`：

$$
\frac{\partial y_n}{\partial y_0}
=
\underbrace{c\times c\times\cdots\times c}_{n\text{ 次}}
=
c^n.
$$

这里的 n 是跨过的 transition 数；在 RNN 中，从 `h_t` 传播到 `h_(T-1)` 一共跨过：

$$
n=(T-1)-t
$$

个 recurrent transitions。

### 当 `c=0.5`

```text
0.5^1  = 0.5
0.5^2  = 0.25
0.5^4  = 0.0625
0.5^8  = 0.00390625
0.5^12 = 0.000244140625
```

越早的变量通过这条长路径对最终结果的 sensitivity 越小。若这个 long-range gradient contribution 小到优化器几乎看不见，我们称之为 **vanishing gradient（梯度消失）**。

### 当 `c=1.5`

```text
1.5^1  = 1.5
1.5^2  = 2.25
1.5^4  = 5.0625
1.5^8  = 25.62890625
1.5^12 ≈ 129.7463
```

长路径 sensitivity 反而快速放大，这就是 **exploding gradient（梯度爆炸）** 的最小直觉。

但是必须立刻加上边界：

> `0.5^n` 和 `1.5^n` 只是“每一步都恰好乘同一个 scalar”的教学模型。真实 RNN 的局部因子会随参数、hidden state、时间步和向量方向变化。

因此本节不会用这两个幂直接替代真实 Jacobian 分析。


In [ ]:
import matplotlib.pyplot as plt

steps = list(range(1, 13))
decay = [0.5 ** n for n in steps]
growth = [1.5 ** n for n in steps]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(steps, decay, marker='o')
axes[0].set_title('Repeated factor 0.5: decay')
axes[0].set_xlabel('number of multiplied local factors n')
axes[0].set_ylabel('0.5^n')
axes[0].grid(True, alpha=0.25)

axes[1].plot(steps, growth, marker='o')
axes[1].set_title('Repeated factor 1.5: growth')
axes[1].set_xlabel('number of multiplied local factors n')
axes[1].set_ylabel('1.5^n')
axes[1].grid(True, alpha=0.25)

plt.show()


这里用两个独立纵轴，是为了避免巨大的增长曲线把很小的衰减值在同一张线性坐标图里压扁。图只展示 repeated-product intuition，不是 RNN benchmark。


## 2. 接回 Scalar Tanh RNN：局部因子同时由 Weight 和 Hidden State 决定

9.1 已经推导：

$$
a_k=w_xx_k+w_hh_{k-1}+b,
\qquad
h_k=\tanh(a_k),
$$

以及局部 recurrent derivative：

$$
\boxed{
\frac{\partial h_k}{\partial h_{k-1}}
=
w_h(1-h_k^2)
}
$$

因此真实 scalar RNN 的每一步并不是固定乘 `w_h`，而是乘：

$$
c_k=w_h(1-h_k^2).
$$

其中：

```text
w_h             -> recurrent weight 的作用
(1 - h_k^2)     -> tanh 在当前 state 对应位置的局部导数
```

### Hidden state 为什么会改变 gradient flow？

因为 `h_k=tanh(a_k)`。

如果 `a_k≈0`，那么 `h_k≈0`：

$$
1-h_k^2\approx1.
$$

tanh 在这一带没有明显压低局部 sensitivity。

如果 `a_k=1`：

$$
h_k\approx0.761594,
\qquad
1-h_k^2\approx0.419974.
$$

如果 `a_k=2`：

$$
h_k\approx0.964028,
\qquad
1-h_k^2\approx0.070651.
$$

这就是 **tanh saturation（tanh 饱和）** 对 gradient flow 的影响：hidden pre-activation 落到曲线两端时，局部 derivative 可能非常小。

例如取 `w_h=0.9`：

```text
a≈0  -> local factor ≈ 0.9 * 1       = 0.9000
a=1  -> local factor ≈ 0.9 * 0.41997 = 0.3780
a=2  -> local factor ≈ 0.9 * 0.07065 = 0.0636
```

同一个 `w_h`，仅仅因为当前 state 不同，局部 gradient transmission 就可能相差一个数量级。

反过来，即使 `|w_h|>1`，也不能立即说这一时刻一定放大。例如 `w_h=1.5`、`a=2`：

$$
|c_k|
\approx
1.5\times0.070651
\approx0.106.
$$

仍然小于 1。

因此 scalar RNN 中也必须同时看 **recurrent weight + activation derivative + 当前 state**，不能只看 `w_h`。


In [ ]:
import math

wh_values = [0.9, 1.5]
a_values = [0.0, 1.0, 2.0]

for wh in wh_values:
    print(f'w_h = {wh}')
    for a in a_values:
        h = math.tanh(a)
        tanh_prime = 1.0 - h ** 2
        local_factor = wh * tanh_prime
        print(
            f'  a={a:.1f}, tanh(a)={h:.6f}, '
            f'tanh_prime={tanh_prime:.6f}, local_factor={local_factor:.6f}'
        )


这里的 `local_factor` 是 scalar RNN 的 `∂h_k/∂h_(k-1)`。它仍然不是整个 BPTT gradient；完整 long-range path 要把不同时间步的这些局部因子连续相乘，而且 parameter gradient 还可能有 local-loss 路径与不同时间步贡献的求和。


## 3. Vector RNN：同一个 Jacobian 可以让一个方向增长、另一个方向衰减

Scalar 情况只有一个方向；Vector RNN 的 hidden state 有 H 个坐标，因此 gradient 是一个向量。

沿用 9.1 的列向量 convention：

$
g_k=\nabla_{h_k}L
\in\mathbb R^{H\times1}.
$

但这里必须区分 **future-path contribution（未来路径贡献）** 和 **total hidden-state gradient（hidden state 的总梯度）**。

先定义 local Jacobian：

$
J_{k,k-1}
=
\frac{\partial h_k}{\partial h_{k-1}}
\in\mathbb R^{H\times H}.
$

如果 loss 只定义在末端，或者我们此刻只追踪“从 $h_k$ 以及更晚 loss 经过 recurrent edge 回传到 $h_{k-1}$”的那一条 future path，那么：

$
\boxed{
g_{k-1}^{\text{future}}
=
J_{k,k-1}^Tg_k
}
$

这里的 **future term（未来项）** 表示：$h_{k-1}$ 先影响 $h_k$，再通过 $h_k$ 影响当前之后的 loss。

如果总 loss 是：

$
L=\sum_{t=0}^{T-1}\ell_t,
$

而且 $h_{k-1}$ 还直接产生自己的 local loss $\ell_{k-1}$，那么总梯度应写成：

$
\boxed{
g_{k-1}
=
\nabla_{h_{k-1}}\ell_{k-1}
+
J_{k,k-1}^Tg_k
}
$

其中：

- $\nabla_{h_{k-1}}\ell_{k-1}$ 是 **local term（局部项）**：当前 hidden state 对自己这个时间位置 loss 的直接影响；
- $J_{k,k-1}^Tg_k$ 是 **future term（未来项）**：当前 hidden state 通过下一状态继续影响未来 losses 的贡献。

因此，后面讨论 Jacobian 连乘导致的 vanishing / exploding，主要是在分析 **future-path / long-range contribution** 怎样沿时间传播；不能把这一个路径贡献自动当成有 local losses 时的全部梯度。

对于 column-vector Vanilla RNN：

$$
a_k=W_xx_k+W_hh_{k-1}+b,
$$

$$
h_k=\tanh(a_k)
$$

是逐元素 tanh。可以把每个 hidden coordinate 的 tanh derivative 放进一个 **diagonal matrix（对角矩阵）**：

$$
D_k
=
\operatorname{diag}
\left(
1-h_{k,1}^2,\ldots,1-h_{k,H}^2
\right).
$$

`diag(d1,d2)` 的意思只是：

$$
\begin{bmatrix}
d_1&0\\
0&d_2
\end{bmatrix}.
$$

于是 local Jacobian 为：

$$
\boxed{
J_{k,k-1}=D_kW_h
}
$$

shape：

$$
(H,H)(H,H)=(H,H).
$$

因此，这条 recurrent edge 传回来的 future-path contribution 是：

$
g_{k-1}^{\text{future}}
=
W_h^TD_kg_k.
$

如果 $\ell_{k-1}$ 也存在，则总梯度仍要再加上 $\nabla_{h_{k-1}}\ell_{k-1}$。

### 为什么“方向”重要？

看一个专门为直觉构造的 2×2 Jacobian：

$$
J=
\begin{bmatrix}
1.5&0\\
0&0.5
\end{bmatrix}.
$$

如果 upstream gradient 指向第一坐标轴：

$$
g=
\begin{bmatrix}1\\0\end{bmatrix},
$$

经过一次 `J^T`：

$$
J^Tg=
\begin{bmatrix}1.5\\0\end{bmatrix}.
$$

重复 n 次，这一方向变成 `1.5^n`。

但如果 gradient 指向第二坐标轴：

$$
g=
\begin{bmatrix}0\\1\end{bmatrix},
$$

则：

$$
J^Tg=
\begin{bmatrix}0\\0.5\end{bmatrix},
$$

重复 n 次变成 `0.5^n`。

**同一个矩阵，可以同时存在容易放大的方向和容易衰减的方向。**

真实 RNN 更复杂：`D_k` 随 hidden state 改变，所以 `J_{k,k-1}` 往往每个时间步都不同；不同 Jacobian 还可能旋转或混合 gradient directions，而且矩阵乘法顺序不可交换。

因此不能把 scalar 的“某个数大于 1 / 小于 1”直接当成所有矩阵 RNN 的完整判据。要做全局矩阵稳定性分析通常需要更进一步的线性代数工具；本课程此处只保留**方向依赖**这个必要直觉。

### 在第一次使用 `l2_norm` 之前：L2 Norm 是什么？

下面的代码要比较两个 gradient directions 经过 Jacobian 后“整体有多大”，所以先定义 **L2 norm（二范数 / Euclidean norm）**。

对于列向量：

$
g=
\begin{bmatrix}
g_1\\
g_2\\
\vdots
\end{bmatrix},
$

L2 norm 定义为：

$
\boxed{
\|g\|_2
=
\sqrt{\sum_i g_i^2}
}
$

输入是一个 gradient vector，输出是一个**非负 scalar**，可以直观理解为这个向量在 Euclidean space 中的长度。

最小例子：

$
g=
\begin{bmatrix}
3\\
4
\end{bmatrix}
$

则：

$
\|g\|_2
=
\sqrt{3^2+4^2}
=
5.
$

所以代码里的：

```python
def l2_norm(v):
    return mx.sqrt(mx.sum(v * v))
```

就是把这个定义直接翻译成 MLX：先逐元素平方、求和，再开平方。

下面的 2×2 代码只比较 **future-path gradient contribution** 在不同方向上的变化，不是在构造包含 local loss 的完整 BPTT total gradient。



In [ ]:
import mlx.core as mx

J = mx.array([
    [1.5, 0.0],
    [0.0, 0.5],
])

g_grow = mx.array([1.0, 0.0])
g_decay = mx.array([0.0, 1.0])

def l2_norm(v):
    # L2 norm: sqrt(sum_i v_i^2), returns one non-negative scalar
    return mx.sqrt(mx.sum(v * v))

for n in range(1, 5):
    g_grow = J.T @ g_grow
    g_decay = J.T @ g_decay
    print(
        f'n={n}: '
        f'grow={g_grow}, ||grow||_2={l2_norm(g_grow)}, '
        f'decay={g_decay}, ||decay||_2={l2_norm(g_decay)}'
    )


## 4. Gradient Norm 与 Gradient Clipping：它能控制“太大”，但不能找回“已经太小”的信号

### 4.1 从单个向量的 L2 Norm 到 Global Gradient Norm

Section 3 已经在第一次使用 `l2_norm` 之前定义了：

$
\|g\|_2=\sqrt{\sum_i g_i^2}.
$

现在把同一个“平方 → 求和 → 开根号”的思想扩展到真实模型。

真实模型的 gradient 往往不是一个 vector，而是对应许多 parameters 的 tree。**Global gradient norm（全局梯度范数）** 把所有 gradient arrays 中的所有元素一起平方求和，再开根号：

$$
\|G\|_{global}
=
\sqrt{
\sum_{\text{all gradient elements }i}
g_i^2
}.
$$

### 4.2 Global-norm clipping 的最小数学例子

假设：

$$
g=[6,8]^T,
\qquad
\|g\|_2=10.
$$

我们规定最大允许 norm：

$$
max\_norm=5.
$$

理想化 clipping scale：

$$
s
=
\min\left(1,\frac{max\_norm}{\|g\|_2}\right)
=
\frac5{10}
=
0.5.
$$

所以：

$$
g_{clipped}=sg=[3,4]^T,
$$

新的 norm 正好是 5。

如果原 norm 本来只有 2，小于 max_norm=5，那么 `s=1`，gradient 不改变。

Global-norm clipping 对整个 gradient tree 使用同一个正比例系数，因此主要是限制 magnitude；它不会重新发明一个新的 gradient direction。

### 4.3 MLX 0.32.2 的真实 API

项目固定 `mlx==0.32.2`。该版本提供：

```python
mlx.optimizers.clip_grad_norm(grads, max_norm)
```

导入：

```python
import mlx.optimizers as optim
```

输入：

```text
grads     = gradient tree / dictionary of gradient arrays
max_norm  = 允许的最大 global norm（非负 scalar 阈值）
```

输出：

```text
clipped_grads = 可能按统一比例缩小后的 gradient tree
total_norm    = clipping 之前计算出的原始 global norm
```

MLX 0.32.2 实现会计算所有 gradients 的平方和、开根号，然后使用近似：

```text
normalizer = min(max_norm / (total_norm + 1e-6), 1)
```

`1e-6` 是一个很小的数，用来避免 total_norm 接近 0 时除法出现数值问题。

下面只是静态教学代码，不在母版中执行。


In [ ]:
import mlx.core as mx
import mlx.optimizers as optim

grads = {
    'example_weight': mx.array([6.0, 8.0])
}

max_norm = 5.0

clipped_grads, total_norm = optim.clip_grad_norm(
    grads,
    max_norm=max_norm,
)

print('original global norm:', total_norm)
print('original gradient:', grads['example_weight'])
print('clipped gradient:', clipped_grads['example_weight'])


### 4.4 Gradient clipping 能缓解什么？

当 exploding gradient 使某一步 optimizer update 非常大时，clipping 可以把整个 gradient magnitude 限制在阈值附近，从而：

- 减少极端 parameter update；
- 降低训练突然剧烈震荡的风险；
- 在一些 recurrent training 中改善数值稳定性。

但是这里的关键词是 **mitigate（缓解）**，不是“根治”。


## 5. Credit Assignment 的真正困难，以及 Clipping 解决不了什么

### 5.1 Vanishing gradient 不等于“整个模型所有 gradient 都是 0”

长期 recurrent path 从较晚 loss 传回很早状态时，其 **future-path contribution** 可能变得极小，但靠近 loss 的 local gradients 仍然可以很大。

如果 $L=\sum_t\ell_t$ 且每个时间步都有 local loss，那么 hidden-state total gradient 本身就包含：

$
g_{k-1}
=
\nabla_{h_{k-1}}\ell_{k-1}
+
J_{k,k-1}^Tg_k,
$

而 parameter 的总 gradient 还会继续汇总多个时间位置和多条路径的 contributions。于是：

> “长期 credit 消失”描述的是远距离影响难以传回，不应该简单等同于“打印出来的整个 parameter gradient tensor 一定接近 0”。

这也是为什么要把 9.1 的 `Σ`（多路径/多时间贡献求和）和本节的 `Π`（一条长 temporal path 上连续相乘）区分开。

### 5.2 Exploding gradient 的实际后果

如果某些 temporal directions 被持续放大，可能造成：

- gradient norm 很大；
- parameter update 过大；
- loss 剧烈震荡；
- 数值溢出，严重时出现非有限值。

但 exploding 也不是“每个 gradient element 都必须一起变大”。Vector/Jacobian 情况下可能只有某些 directions 被强烈放大。

### 5.3 Clipping 不能解决的事情

Gradient clipping **不能**：

- 把已经 vanishing 的 long-range gradient 恢复回来；
- 消除 tanh saturation；
- 改变 `h_(t-1) -> h_t` 的长 recurrent dependency path；
- 保证模型能记住非常久以前的信息；
- 保证训练一定收敛；
- 在 gradient 已经产生 NaN / Inf 之后自动恢复有意义的信息。

它主要是在 gradient 已经算出来以后，限制过大的 update signal。

### 5.4 为什么状态与方向决定长期 Credit Assignment

Scalar tanh RNN 的 local factor 依赖 `h_k`；Vector RNN 又增加了 gradient direction 与每步 Jacobian 的作用。因此长期 credit assignment 的质量取决于：

```text
recurrent parameters
+ current hidden states
+ activation derivatives
+ gradient direction
+ ordered product of time-varying Jacobians
+ loss 到各状态的路径结构
```

这就是为什么不能只背：

```text
|w_h| < 1 -> vanish
|w_h| > 1 -> explode
```

这样的标量口号。它可以帮助构造某些 toy intuition，但不是一般 vector RNN 的完整判据。

### Step 9.2 材料边界

- `0.5^n / 1.5^n` 只用于 repeated scalar factor 的直觉。
- scalar tanh RNN 的局部因子是 `w_h(1-h_k^2)`，所以 hidden state / saturation 会改变传递强度。
- vector RNN 使用 `(H,H)` Jacobian；同一个 Jacobian 可以缩小某些 directions、放大另一些 directions。
- L2 norm `||g||_2` 把 gradient vector 的 magnitude 压成非负 scalar；global norm 把整个 gradient tree 的元素一起纳入。
- MLX 0.32.2 `optim.clip_grad_norm(grads, max_norm)` 返回 clipped gradient tree 与原始 total norm。
- gradient clipping 可以限制 exploding-gradient 的 magnitude，但不能修复 vanishing gradient、tanh saturation 或长期记忆机制。
- 本节只准备 **Step 9.2**；不进入 Step 9.3，也不声明学员已经掌握或完成本节练习。
